# BirdCLEF 2026: Train + Validate + Submit (Colab)

This notebook runs the implemented pipeline from the repository using Colab.

It covers:
- environment setup
- data checks
- CV split + leakage checks
- baseline training (and optional alt model family)
- OOF threshold optimization
- submission schema/runtime checks

Note: Kaggle hidden test soundscapes are available only during Kaggle scoring runs.

## 0) Configuration

Edit these variables before running all cells.

In [30]:
# User-configurable settings
REPO_DIR = '/content/drive/MyDrive/repos/competitions/birdclef-2026'  # final repo working path in Colab
DRIVE_ROOT = '/content/drive/MyDrive/birdclef-2026'

# Optional git-clone settings (recommended for fresh Colab sessions)
GIT_CLONE_AT_START = True
GIT_REPO_URL = 'https://github.com/neilken/competitions.git'
GIT_BRANCH = 'main'
GIT_PULL_IF_EXISTS = True
GITHUB_TOKEN_ENV_VAR = 'GITHUB_TOKEN'  # used only for private repo HTTPS auth

# Clone location controls
CLONE_REPO_TO_DRIVE = True  # when True, clone into Google Drive so sessions can reuse it
DRIVE_CLONE_ROOT = '/content/drive/MyDrive/repos/competitions'
LOCAL_CLONE_ROOT = '/content/competitions'

# Optional sparse checkout settings (clone only one folder from repo)
USE_SPARSE_CHECKOUT = True
SPARSE_FOLDER = 'birdclef-2026'  # folder path inside repo

# Optional toggles
RUN_FAST_BASELINE = True
RUN_ALT_MODEL_FAMILY = False
RUN_ABLATIONS = False
RUN_AUDIO_SMOKE_TEST = True

BASELINE_CONFIG_PATH = 'configs/baseline_colab_fast.yaml' if RUN_FAST_BASELINE else 'configs/baseline_colab.yaml'
ALT_CONFIG_PATH = 'configs/baseline_alt_colab_fast.yaml' if RUN_FAST_BASELINE else 'configs/baseline_alt_colab.yaml'
BASELINE_CONFIG_ID = 'baseline_colab_fast_v1' if RUN_FAST_BASELINE else 'baseline_colab_v1'

# Optional: If you copied test soundscapes into Drive for dry-run inference, set this.
LOCAL_TEST_SOUNDSCAPE_DIR = ''  # e.g. '/content/drive/MyDrive/birdclef-2026/data/test_soundscapes_local'

print('[CONFIG] REPO_DIR=', REPO_DIR)
print('[CONFIG] DRIVE_ROOT=', DRIVE_ROOT)
print('[CONFIG] GIT_CLONE_AT_START=', GIT_CLONE_AT_START)
print('[CONFIG] GIT_REPO_URL=', GIT_REPO_URL or '<not set>')
print('[CONFIG] GIT_BRANCH=', GIT_BRANCH)
print('[CONFIG] GIT_PULL_IF_EXISTS=', GIT_PULL_IF_EXISTS)
print('[CONFIG] GITHUB_TOKEN_ENV_VAR=', GITHUB_TOKEN_ENV_VAR)
print('[CONFIG] CLONE_REPO_TO_DRIVE=', CLONE_REPO_TO_DRIVE)
print('[CONFIG] DRIVE_CLONE_ROOT=', DRIVE_CLONE_ROOT)
print('[CONFIG] LOCAL_CLONE_ROOT=', LOCAL_CLONE_ROOT)
print('[CONFIG] USE_SPARSE_CHECKOUT=', USE_SPARSE_CHECKOUT)
print('[CONFIG] SPARSE_FOLDER=', SPARSE_FOLDER)
print('[CONFIG] RUN_FAST_BASELINE=', RUN_FAST_BASELINE)
print('[CONFIG] BASELINE_CONFIG_PATH=', BASELINE_CONFIG_PATH)
print('[CONFIG] BASELINE_CONFIG_ID=', BASELINE_CONFIG_ID)
print('[CONFIG] ALT_CONFIG_PATH=', ALT_CONFIG_PATH)
print('[CONFIG] RUN_ALT_MODEL_FAMILY=', RUN_ALT_MODEL_FAMILY)
print('[CONFIG] RUN_ABLATIONS=', RUN_ABLATIONS)
print('[CONFIG] RUN_AUDIO_SMOKE_TEST=', RUN_AUDIO_SMOKE_TEST)
print('[CONFIG] LOCAL_TEST_SOUNDSCAPE_DIR=', LOCAL_TEST_SOUNDSCAPE_DIR or '<not set>')

[CONFIG] REPO_DIR= /content/drive/MyDrive/repos/competitions/birdclef-2026
[CONFIG] DRIVE_ROOT= /content/drive/MyDrive/birdclef-2026
[CONFIG] GIT_CLONE_AT_START= True
[CONFIG] GIT_REPO_URL= https://github.com/neilken/competitions.git
[CONFIG] GIT_BRANCH= main
[CONFIG] GIT_PULL_IF_EXISTS= True
[CONFIG] GITHUB_TOKEN_ENV_VAR= GITHUB_TOKEN
[CONFIG] CLONE_REPO_TO_DRIVE= True
[CONFIG] DRIVE_CLONE_ROOT= /content/drive/MyDrive/repos/competitions
[CONFIG] LOCAL_CLONE_ROOT= /content/competitions
[CONFIG] USE_SPARSE_CHECKOUT= True
[CONFIG] SPARSE_FOLDER= birdclef-2026
[CONFIG] RUN_FAST_BASELINE= True
[CONFIG] BASELINE_CONFIG_PATH= configs/baseline_colab_fast.yaml
[CONFIG] BASELINE_CONFIG_ID= baseline_colab_fast_v1
[CONFIG] ALT_CONFIG_PATH= configs/baseline_alt_colab_fast.yaml
[CONFIG] RUN_ALT_MODEL_FAMILY= False
[CONFIG] RUN_ABLATIONS= False
[CONFIG] RUN_AUDIO_SMOKE_TEST= True
[CONFIG] LOCAL_TEST_SOUNDSCAPE_DIR= <not set>


## 1) Mount Drive and Helper Runner

In [31]:
from google.colab import drive
from collections import deque
import os
import re
import shlex
import subprocess
import time
from pathlib import Path

print('[STEP] Mounting Google Drive...')
drive.mount('/content/drive')
print('[OK] Drive mounted.')

def q(path_like):
    """Shell-quote helper for paths/URLs in run() commands."""
    return shlex.quote(str(path_like))

def redact_cmd(cmd: str) -> str:
    """Hide inline URL credentials if present in logged shell commands."""
    return re.sub(r'(https://)([^/@\\s]+)@', r'\\1***@', cmd)

def run(cmd, cwd=None, check=True):
    """Run a shell command, stream stdout/stderr live, and raise with tail on failure."""
    safe_cmd = redact_cmd(cmd)
    print('\n' + '=' * 88)
    print('[RUN]', safe_cmd)
    if cwd:
      print('[CWD]', cwd)
    print('=' * 88)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )
    tail_lines = deque(maxlen=120)
    if proc.stdout is not None:
        for line in proc.stdout:
            print(line, end='')
            tail_lines.append(line.rstrip('\n'))
    proc.wait()
    dt = time.time() - t0
    print(f'[DONE] exit_code={proc.returncode} elapsed_sec={dt:.2f}')
    if check and proc.returncode != 0:
        tail = '\n'.join(tail_lines)
        raise RuntimeError(
            f'Command failed: {safe_cmd}\n'
            f'Exit code: {proc.returncode}\n'
            f'----- Last output lines -----\n{tail}'
        )
    return proc.returncode


[STEP] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Drive mounted.


## 2) Repository and Dependencies

In [32]:
repo_path = Path(REPO_DIR)
clone_root = Path(DRIVE_CLONE_ROOT if CLONE_REPO_TO_DRIVE else LOCAL_CLONE_ROOT)

if GIT_CLONE_AT_START:
    print('[STEP] GIT_CLONE_AT_START enabled. Preparing clone flow...')
    if not GIT_REPO_URL:
        raise ValueError('Set GIT_REPO_URL in the config cell when GIT_CLONE_AT_START=True.')

    clone_url = GIT_REPO_URL
    token = os.environ.get(GITHUB_TOKEN_ENV_VAR, '').strip()
    if token and 'github.com' in GIT_REPO_URL and GIT_REPO_URL.startswith('https://'):
        # Private repo helper: use token in HTTPS URL for this clone operation.
        clone_url = GIT_REPO_URL.replace('https://', f'https://{token}@', 1)
        print(f'[INFO] Using token from env var {GITHUB_TOKEN_ENV_VAR} for authenticated clone.')

    clone_root.parent.mkdir(parents=True, exist_ok=True)
    git_dir = clone_root / '.git'

    if git_dir.exists():
        print(f'[INFO] Existing clone found. Reusing: {clone_root}')
        if USE_SPARSE_CHECKOUT:
            run('git sparse-checkout init --cone', cwd=str(clone_root))
            run(f'git sparse-checkout set {q(SPARSE_FOLDER)}', cwd=str(clone_root))
        run(f'git checkout {q(GIT_BRANCH)}', cwd=str(clone_root))
        if GIT_PULL_IF_EXISTS:
            run(f'git pull --ff-only origin {q(GIT_BRANCH)}', cwd=str(clone_root))
        else:
            print('[SKIP] GIT_PULL_IF_EXISTS=False, using local clone state as-is.')
    elif clone_root.exists():
        raise RuntimeError(
            f'Clone root exists but is not a git repo: {clone_root}. '
            'Use a different clone root or remove this directory manually.'
        )
    else:
        if USE_SPARSE_CHECKOUT:
            print('[INFO] Running sparse checkout clone flow...')
            run(f'git clone --filter=blob:none --no-checkout {q(clone_url)} {q(clone_root)}')
            run('git sparse-checkout init --cone', cwd=str(clone_root))
            run(f'git sparse-checkout set {q(SPARSE_FOLDER)}', cwd=str(clone_root))
            run(f'git checkout {q(GIT_BRANCH)}', cwd=str(clone_root))
        else:
            print('[INFO] Running full clone flow...')
            run(f'git clone --branch {q(GIT_BRANCH)} {q(clone_url)} {q(clone_root)}')

    if USE_SPARSE_CHECKOUT:
        repo_path = clone_root / SPARSE_FOLDER
    else:
        repo_path = clone_root

    REPO_DIR = str(repo_path)
    print(f'[OK] Repo cloned and ready at: {repo_path}')

if not repo_path.exists():
    print(f'[ERROR] Repo not found at {repo_path}')
    print('Either set GIT_CLONE_AT_START=True with GIT_REPO_URL, or copy the repo into Colab manually.')
    raise FileNotFoundError(repo_path)

print(f'[OK] Repo ready: {repo_path}')
run('python --version', cwd=str(repo_path))
run('pip install -r requirements-colab.txt', cwd=str(repo_path))

[STEP] GIT_CLONE_AT_START enabled. Preparing clone flow...
[INFO] Existing clone found. Reusing: /content/drive/MyDrive/repos/competitions

[RUN] git sparse-checkout init --cone
[CWD] /content/drive/MyDrive/repos/competitions
[DONE] exit_code=0 elapsed_sec=0.22

[RUN] git sparse-checkout set birdclef-2026
[CWD] /content/drive/MyDrive/repos/competitions
[DONE] exit_code=0 elapsed_sec=0.04

[RUN] git checkout main
[CWD] /content/drive/MyDrive/repos/competitions
Already on 'main'
Your branch is up to date with 'origin/main'.
[DONE] exit_code=0 elapsed_sec=0.21

[RUN] git pull --ff-only origin main
[CWD] /content/drive/MyDrive/repos/competitions
From https://github.com/neilken/competitions
 * branch            main       -> FETCH_HEAD
Already up to date.
[DONE] exit_code=0 elapsed_sec=0.41
[OK] Repo cloned and ready at: /content/drive/MyDrive/repos/competitions/birdclef-2026
[OK] Repo ready: /content/drive/MyDrive/repos/competitions/birdclef-2026

[RUN] python --version
[CWD] /content/driv

0

## 3) Data Integrity and Smoke Checks

In [33]:
data_root = f'{DRIVE_ROOT}/data'
raw_archive = f'{DRIVE_ROOT}/raw/kaggle-download/birdclef-2026.zip'

run(
    f'python -u -m src.utils.data_integrity --data-root {q(data_root)} --raw-archive {q(raw_archive)}',
    cwd=REPO_DIR,
)

if RUN_AUDIO_SMOKE_TEST:
    run(
        'python -u -m src.utils.audio_smoke_test '
        f'--train-audio-dir {q(f"{data_root}/train_audio")} '
        f'--train-soundscapes-dir {q(f"{data_root}/train_soundscapes")} '
        '--samples-per-dir 10',
        cwd=REPO_DIR,
    )
else:
    print('[SKIP] Audio smoke test disabled by config.')


[RUN] python -u -m src.utils.data_integrity --data-root /content/drive/MyDrive/birdclef-2026/data --raw-archive /content/drive/MyDrive/birdclef-2026/raw/kaggle-download/birdclef-2026.zip
[CWD] /content/drive/MyDrive/repos/competitions/birdclef-2026
[STEP 1/4] Validating configured data root...
[STEP 2/4] Checking required entries with progress bar...

data entries:   0%|          | 0/7 [00:00<?, ?entry/s][OK] train_audio: 206 direct entries
[OK] train_soundscapes: 10658 direct entries

data entries:  29%|██▊       | 2/7 [00:00<00:00, 14.91entry/s][OK] train.csv: 6779071 bytes
[OK] taxonomy.csv: 13693 bytes
[OK] sample_submission.csv: 16708 bytes
[OK] train_soundscapes_labels.csv: 134623 bytes
[OK] recording_location.txt: 204 bytes

data entries: 100%|██████████| 7/7 [00:00<00:00, 51.05entry/s]
[STEP 3/4] Core dataset entries validated.
[OK] Raw archive found: /content/drive/MyDrive/birdclef-2026/raw/kaggle-download/birdclef-2026.zip (16073652829 bytes)
[STEP 4/4] Data integrity checks

## 4) Build CV Splits and Validate Leakage

In [ ]:
run('python -u -m src.training.create_cv_splits --config configs/cv_policy.yaml', cwd=REPO_DIR)
run(
    f'python -u -m src.training.check_fold_leakage --folds-csv {q(f"{DRIVE_ROOT}/outputs/reports/folds.csv")}',
    cwd=REPO_DIR,
)


[RUN] python -u -m src.training.create_cv_splits --config configs/cv_policy.yaml
[CWD] /content/drive/MyDrive/repos/competitions/birdclef-2026
[STEP 1/6] Loading CV config...
[INFO] config=configs/cv_policy.yaml
[INFO] labels_csv=/content/drive/MyDrive/birdclef-2026/data/train_soundscapes_labels.csv
[INFO] folds_csv=/content/drive/MyDrive/birdclef-2026/outputs/reports/folds.csv
[INFO] n_splits=5
[STEP 2/6] Reading labels CSV...
[STEP 3/6] Building row_id and group_id fields...

group_id: 100%|██████████| 1478/1478 [00:00<00:00, 137974.21it/s]
[STEP 4/6] Assigning GroupKFold splits...

fold assignment: 100%|██████████| 5/5 [00:00<00:00, 845.93fold/s]
[STEP 5/6] Writing folds CSV...
[STEP 6/6] Reporting split summary...
[OK] Wrote folds to: /content/drive/MyDrive/birdclef-2026/outputs/reports/folds.csv
[INFO] Rows: 1478 | Groups: 53 | Fallback parse count: 0
[INFO] Fold distribution:
fold
0    300
1    298
2    294
3    288
4    298
Name: count, dtype: int64
[DONE] exit_code=0 elapsed_s

0

## 5) Train Baseline Model Family

In [ ]:
run(f'python -u -m src.training.run_baseline --config {q(BASELINE_CONFIG_PATH)}', cwd=REPO_DIR)


[RUN] python -u -m src.training.run_baseline --config configs/baseline_colab.yaml
[CWD] /content/drive/MyDrive/repos/competitions/birdclef-2026

[STEP 1/8] Loading run configuration

[STEP 2/8] Seeding and device selection
[INFO] Training device: cuda

[STEP 3/8] Resolving input/output paths
[INFO] data_root=/content/drive/MyDrive/birdclef-2026/data
[INFO] output_root=/content/drive/MyDrive/birdclef-2026/outputs
[INFO] folds_csv=/content/drive/MyDrive/birdclef-2026/outputs/reports/folds.csv
[INFO] registry_csv=/content/drive/MyDrive/birdclef-2026/docs/trackers/experiment_registry.csv

[STEP 4/8] Loading metadata and fold definitions
[INFO] taxonomy rows=234
[INFO] fold rows=1478
[INFO] num classes=234
[INFO] training config summary:
  folds=5, epochs=5, batch_size=24, lr=0.0001
  requested_num_workers=2, resolved_num_workers=2
  sample_rate=32000, clip_seconds=5.0, n_mels=128, n_fft=2048, hop=512

[STEP 5/8] Fold training loop

folds:   0%|          | 0/5 [00:00<?, ?fold/s][INFO] fold

KeyboardInterrupt: 

## 6) Train Alternative Model Family (Optional)

In [ ]:
if RUN_ALT_MODEL_FAMILY:
    run(f'python -u -m src.training.run_baseline --config {q(ALT_CONFIG_PATH)}', cwd=REPO_DIR)
else:
    print('[SKIP] Alternative model family disabled by config.')

## 7) Run Ablation Matrix (Optional)

In [ ]:
if RUN_ABLATIONS:
    run(f'python -u -m src.training.run_ablations --base-config {q(BASELINE_CONFIG_PATH)} --ablation-config configs/ablations.yaml', cwd=REPO_DIR)
else:
    print('[SKIP] Ablations disabled by config.')

## 8) Optimize Class Thresholds from OOF

In [ ]:
run(
    'python -u -m src.training.optimize_thresholds '
    f'--oof-csv {q(f"{DRIVE_ROOT}/outputs/oof/{BASELINE_CONFIG_ID}_oof.csv")} '
    f'--folds-csv {q(f"{DRIVE_ROOT}/outputs/reports/folds.csv")} '
    f'--output-csv {q(f"{DRIVE_ROOT}/outputs/reports/class_thresholds.csv")}',
    cwd=REPO_DIR,
)

## 9) Generate Submission Dry-Run

If `LOCAL_TEST_SOUNDSCAPE_DIR` is set and contains `.ogg` files + row_id compatibility, the notebook will run checkpoint inference.

Otherwise it writes a schema-valid zero baseline `submission.csv` for pipeline checks.

In [ ]:
sample_sub = f'{DRIVE_ROOT}/data/sample_submission.csv'
submission_out = f'{DRIVE_ROOT}/outputs/submissions/submission.csv'
checkpoint = f'{DRIVE_ROOT}/outputs/checkpoints/{BASELINE_CONFIG_ID}_fold0.pt'

if LOCAL_TEST_SOUNDSCAPE_DIR:
    cmd = (
        'python -u -m src.inference.generate_submission_cpu '
        f'--sample-submission {q(sample_sub)} '
        f'--checkpoint {q(checkpoint)} '
        f'--soundscape-dir {q(LOCAL_TEST_SOUNDSCAPE_DIR)} '
        f'--output {q(submission_out)}'
    )
else:
    cmd = (
        'python -u -m src.inference.generate_submission_cpu '
        f'--sample-submission {q(sample_sub)} '
        f'--output {q(submission_out)}'
    )

run(cmd, cwd=REPO_DIR)

## 10) Validate Submission + Rules Gate + Runtime Rehearsal

In [ ]:
run(
    'python -u -m src.inference.validate_submission '
    f'--sample-submission {q(f"{DRIVE_ROOT}/data/sample_submission.csv")} '
    f'--submission {q(f"{DRIVE_ROOT}/outputs/submissions/submission.csv")}',
    cwd=REPO_DIR,
)

run('python -u -m src.utils.metric_parity_check', cwd=REPO_DIR)
run('python -u -m src.utils.determinism_check', cwd=REPO_DIR)
run('python -u -m src.utils.rules_gate --config configs/rules_gate.yaml', cwd=REPO_DIR)

# Runtime rehearsal runs configured submission generation and checks budget.
run('python -u -m src.inference.runtime_rehearsal --config configs/inference_cpu_submission.yaml --max-minutes 90', cwd=REPO_DIR)

## 11) Log Submission Attempt

In [ ]:
run(
    'python -u -m src.utils.submission_log '
    f'--log-csv {q(f"{DRIVE_ROOT}/docs/trackers/submission_log.csv")} '
    f'--set submission_id=LOCAL_DRY_RUN run_id=colab_pipeline config_id={BASELINE_CONFIG_ID} '
    'is_final_candidate=false rules_gate_passed=true notes="colab dry run"',
    cwd=REPO_DIR,
)

## 12) Kaggle Final-Scoring Note

Final competition scoring must happen in a Kaggle Notebook environment where hidden test soundscapes are mounted.

Use this repository code there with CPU + no internet + `submission.csv` output.